# 04 - Analysis

Produces the key figures and tables:
1. Accuracy table with 95% bootstrap CIs
2. Layer sweep plot (the key figure)
3. Per-problem analysis (where zeroed_layer_0 fails but normal succeeds)
4. KL divergence heatmap

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/10-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, INTERVENTION_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.2)
%matplotlib inline

In [ ]:
# Load all results
def load_results(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines()]

# Normal results (from COT generation)
cots = load_results(RESULTS_DIR / "cots.jsonl")
print(f"Normal (COT): {len(cots)} entries")

# Intervention results
results = {}
for cond in CONDITIONS:
    entries = load_results(RESULTS_DIR / f"{cond}.jsonl")
    if entries:
        results[cond] = entries
        print(f"{cond}: {len(entries)} entries")
    else:
        print(f"{cond}: NOT FOUND")

In [ ]:
# Compute accuracy with bootstrap confidence intervals
def accuracy_with_ci(entries, n_bootstrap=10000, ci=0.95):
    """Compute accuracy and bootstrap 95% CI."""
    valid = [e for e in entries if e.get("error") is None]
    if not valid:
        return 0, 0, 0, 0
    
    correct = np.array([1 if e["predicted_answer"] == e["gold_answer"] else 0 for e in valid])
    acc = correct.mean()
    
    # Bootstrap
    rng = np.random.default_rng(42)
    boot_accs = []
    for _ in range(n_bootstrap):
        sample = rng.choice(correct, size=len(correct), replace=True)
        boot_accs.append(sample.mean())
    
    boot_accs = np.array(boot_accs)
    alpha = (1 - ci) / 2
    lo = np.quantile(boot_accs, alpha)
    hi = np.quantile(boot_accs, 1 - alpha)
    
    return acc, lo, hi, len(valid)

In [ ]:
# 1. Accuracy table
print("="*70)
print(f"{'Condition':<25} {'Accuracy':>10} {'95% CI':>20} {'N':>6}")
print("="*70)

# Normal
acc, lo, hi, n = accuracy_with_ci(cots)
table_rows = [("normal", acc, lo, hi, n)]
print(f"{'normal':<25} {acc:>10.1%} [{lo:.1%}, {hi:.1%}] {n:>6}")

# Intervention conditions
for cond in CONDITIONS:
    if cond in results:
        acc, lo, hi, n = accuracy_with_ci(results[cond])
        table_rows.append((cond, acc, lo, hi, n))
        print(f"{cond:<25} {acc:>10.1%} [{lo:.1%}, {hi:.1%}] {n:>6}")
    else:
        print(f"{cond:<25} {'N/A':>10}")

print("="*70)

In [ ]:
# 2. Layer sweep plot (KEY FIGURE)
layer_indices = []
layer_accs = []
layer_ci_lo = []
layer_ci_hi = []

for k in ZERO_AT_LAYERS:
    cond = f"zeroed_layer_{k}"
    if cond in results:
        acc, lo, hi, _ = accuracy_with_ci(results[cond])
        layer_indices.append(k)
        layer_accs.append(acc)
        layer_ci_lo.append(lo)
        layer_ci_hi.append(hi)

if layer_indices:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Normal accuracy baseline
    normal_acc, normal_lo, normal_hi, _ = accuracy_with_ci(cots)
    ax.axhspan(normal_lo, normal_hi, alpha=0.15, color="green", label="Normal 95% CI")
    ax.axhline(normal_acc, color="green", linestyle="--", linewidth=1.5, label=f"Normal ({normal_acc:.1%})")
    
    # Self-prefill baseline
    if "self_prefill" in results:
        sp_acc, sp_lo, sp_hi, _ = accuracy_with_ci(results["self_prefill"])
        ax.axhline(sp_acc, color="blue", linestyle=":", linewidth=1.5, label=f"Self-prefill ({sp_acc:.1%})")
    
    # Layer sweep
    layer_accs_arr = np.array(layer_accs)
    layer_ci_lo_arr = np.array(layer_ci_lo)
    layer_ci_hi_arr = np.array(layer_ci_hi)
    
    ax.errorbar(
        layer_indices, layer_accs_arr,
        yerr=[layer_accs_arr - layer_ci_lo_arr, layer_ci_hi_arr - layer_accs_arr],
        marker="o", markersize=8, linewidth=2, capsize=5,
        color="red", label="Zeroed residual"
    )
    
    ax.set_xlabel("Layer index where residual is zeroed")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"COT Faithfulness: Effect of Residual Stream Zeroing\n{MODEL_NAME} on GSM8K")
    ax.set_xticks(ZERO_AT_LAYERS)
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left")
    
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "layer_sweep.png", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "layer_sweep.pdf", bbox_inches="tight")
    print(f"Saved to {FIGURES_DIR / 'layer_sweep.png'}")
    plt.show()
else:
    print("No zeroed_layer results available yet.")

In [ ]:
# 3. Per-problem analysis: where does zeroed_layer_0 fail but normal succeeds?
if "zeroed_layer_0" in results:
    # Build lookup by problem_id
    normal_by_id = {e["problem_id"]: e for e in cots if e.get("error") is None}
    zeroed_by_id = {e["problem_id"]: e for e in results["zeroed_layer_0"] if e.get("error") is None}
    
    # Problems where normal is correct but zeroed_layer_0 is wrong
    subliminal_failures = []
    for pid in normal_by_id:
        if pid not in zeroed_by_id:
            continue
        n = normal_by_id[pid]
        z = zeroed_by_id[pid]
        if n["predicted_answer"] == n["gold_answer"] and z["predicted_answer"] != z["gold_answer"]:
            subliminal_failures.append({
                "problem_id": pid,
                "question": n["question"],
                "gold_answer": n["gold_answer"],
                "normal_answer": n["predicted_answer"],
                "zeroed_answer": z["predicted_answer"],
                "zeroed_top1_token": z.get("top1_token"),
                "zeroed_top1_prob": z.get("top1_prob"),
                "cot_text": n.get("cot_text", ""),
            })
    
    common_ids = set(normal_by_id.keys()) & set(zeroed_by_id.keys())
    normal_correct_ids = {pid for pid in common_ids if normal_by_id[pid]["predicted_answer"] == normal_by_id[pid]["gold_answer"]}
    
    print(f"Problems where normal is correct: {len(normal_correct_ids)}")
    print(f"Of those, zeroed_layer_0 fails: {len(subliminal_failures)}")
    if normal_correct_ids:
        print(f"Subliminal failure rate: {len(subliminal_failures)/len(normal_correct_ids):.1%}")
    
    # Show a few examples
    print("\n--- Example subliminal failures ---")
    for sf in subliminal_failures[:5]:
        print(f"\nProblem {sf['problem_id']}:")
        print(f"  Q: {sf['question'][:120]}...")
        print(f"  Gold: {sf['gold_answer']} | Normal: {sf['normal_answer']} | Zeroed: {sf['zeroed_answer']}")
        print(f"  Zeroed top-1: '{sf['zeroed_top1_token']}' (p={sf['zeroed_top1_prob']})")
        cot_len = len(sf['cot_text'].split()) if sf['cot_text'] else 0
        print(f"  COT length: {cot_len} words")
    
    # COT length analysis: are subliminal failures associated with shorter COTs?
    if subliminal_failures:
        fail_cot_lengths = [len(sf["cot_text"].split()) for sf in subliminal_failures if sf["cot_text"]]
        all_cot_lengths = [len(n["cot_text"].split()) for n in normal_by_id.values() if n.get("cot_text")]
        
        print(f"\nCOT length (subliminal failures): mean={np.mean(fail_cot_lengths):.0f}, median={np.median(fail_cot_lengths):.0f}")
        print(f"COT length (all problems):         mean={np.mean(all_cot_lengths):.0f}, median={np.median(all_cot_lengths):.0f}")
else:
    print("zeroed_layer_0 results not available yet.")

In [ ]:
# 4. KL divergence analysis
# We need to compute KL from the stored logits_top10 (approximate)
# For a more precise KL, we'd need full logit vectors, but top-10 gives a useful signal

# Instead, we use gold_token_rank and top1_prob as proxies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean top-1 confidence per condition
cond_names = []
mean_confidences = []
ci_lo_conf = []
ci_hi_conf = []

for cond in CONDITIONS:
    if cond not in results:
        continue
    valid = [e for e in results[cond] if e.get("error") is None and e.get("top1_prob") is not None]
    if not valid:
        continue
    probs = np.array([e["top1_prob"] for e in valid])
    cond_names.append(cond.replace("zeroed_layer_", "L").replace("self_prefill", "self-prefill"))
    mean_confidences.append(probs.mean())
    # Bootstrap CI for mean
    rng = np.random.default_rng(42)
    boots = [rng.choice(probs, size=len(probs), replace=True).mean() for _ in range(5000)]
    ci_lo_conf.append(np.quantile(boots, 0.025))
    ci_hi_conf.append(np.quantile(boots, 0.975))

if cond_names:
    mean_conf_arr = np.array(mean_confidences)
    ci_lo_arr = np.array(ci_lo_conf)
    ci_hi_arr = np.array(ci_hi_conf)
    
    axes[0].barh(
        cond_names, mean_conf_arr,
        xerr=[mean_conf_arr - ci_lo_arr, ci_hi_arr - mean_conf_arr],
        capsize=4, color=sns.color_palette("coolwarm", len(cond_names))
    )
    axes[0].set_xlabel("Mean top-1 probability")
    axes[0].set_title("Top-1 Confidence by Condition")

# Median gold token rank per condition
cond_names2 = []
median_ranks = []
ci_lo_rank = []
ci_hi_rank = []

for cond in CONDITIONS:
    if cond not in results:
        continue
    valid = [e for e in results[cond] if e.get("error") is None and e.get("gold_token_rank") is not None]
    if not valid:
        continue
    ranks = np.array([e["gold_token_rank"] for e in valid])
    cond_names2.append(cond.replace("zeroed_layer_", "L").replace("self_prefill", "self-prefill"))
    median_ranks.append(np.median(ranks))
    rng = np.random.default_rng(42)
    boots = [np.median(rng.choice(ranks, size=len(ranks), replace=True)) for _ in range(5000)]
    ci_lo_rank.append(np.quantile(boots, 0.025))
    ci_hi_rank.append(np.quantile(boots, 0.975))

if cond_names2:
    med_arr = np.array(median_ranks)
    ci_lo_r = np.array(ci_lo_rank)
    ci_hi_r = np.array(ci_hi_rank)
    
    axes[1].barh(
        cond_names2, med_arr,
        xerr=[med_arr - ci_lo_r, ci_hi_r - med_arr],
        capsize=4, color=sns.color_palette("coolwarm", len(cond_names2))
    )
    axes[1].set_xlabel("Median gold token rank")
    axes[1].set_title("Gold Token Rank by Condition")
    axes[1].invert_xaxis()  # Lower rank is better

plt.tight_layout()
fig.savefig(FIGURES_DIR / "confidence_and_rank.png", dpi=150, bbox_inches="tight")
fig.savefig(FIGURES_DIR / "confidence_and_rank.pdf", bbox_inches="tight")
print(f"Saved to {FIGURES_DIR / 'confidence_and_rank.png'}")
plt.show()

In [ ]:
# 5. Accuracy drop heatmap: per-layer accuracy delta from normal
if len(layer_indices) > 0:
    normal_acc, _, _, _ = accuracy_with_ci(cots)
    
    deltas = [acc - normal_acc for acc in layer_accs]
    
    fig, ax = plt.subplots(figsize=(10, 2))
    delta_arr = np.array(deltas).reshape(1, -1)
    
    im = ax.imshow(delta_arr, cmap="RdYlGn", aspect="auto", vmin=-0.5, vmax=0.1)
    ax.set_xticks(range(len(layer_indices)))
    ax.set_xticklabels([f"L{k}" for k in layer_indices])
    ax.set_yticks([])
    ax.set_title("Accuracy delta from normal (green = no drop, red = large drop)")
    
    # Annotate with values
    for i, d in enumerate(deltas):
        ax.text(i, 0, f"{d:+.1%}", ha="center", va="center", fontsize=12, fontweight="bold")
    
    plt.colorbar(im, ax=ax, label="Accuracy delta", shrink=0.8)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / "accuracy_delta_heatmap.png", dpi=150, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / "accuracy_delta_heatmap.pdf", bbox_inches="tight")
    print(f"Saved to {FIGURES_DIR / 'accuracy_delta_heatmap.png'}")
    plt.show()
else:
    print("No layer sweep data available yet.")

In [ ]:
# 6. Summary statistics table
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"Model: {MODEL_NAME}")
print(f"Dataset: GSM8K ({DATASET_SPLIT})")
print()
print(f"{'Condition':<25} {'Acc':>8} {'95% CI':>20} {'Mean P(top1)':>14} {'Med Rank(gold)':>16} {'N':>6}")
print("-"*80)

# Normal
acc, lo, hi, n = accuracy_with_ci(cots)
print(f"{'normal':<25} {acc:>8.1%} [{lo:.1%}, {hi:.1%}] {'N/A':>14} {'N/A':>16} {n:>6}")

for cond in CONDITIONS:
    if cond not in results:
        continue
    acc, lo, hi, n = accuracy_with_ci(results[cond])
    valid = [e for e in results[cond] if e.get("error") is None]
    
    probs = [e["top1_prob"] for e in valid if e.get("top1_prob") is not None]
    ranks = [e["gold_token_rank"] for e in valid if e.get("gold_token_rank") is not None]
    
    mean_p = np.mean(probs) if probs else float('nan')
    med_rank = np.median(ranks) if ranks else float('nan')
    
    print(f"{cond:<25} {acc:>8.1%} [{lo:.1%}, {hi:.1%}] {mean_p:>14.3f} {med_rank:>16.1f} {n:>6}")

print("="*80)

In [ ]:
# 7. Key finding
if "zeroed_layer_0" in results:
    normal_acc, _, _, _ = accuracy_with_ci(cots)
    z0_acc, _, _, _ = accuracy_with_ci(results["zeroed_layer_0"])
    delta = z0_acc - normal_acc
    
    print("KEY FINDING:")
    print(f"Normal accuracy:          {normal_acc:.1%}")
    print(f"Zeroed layer 0 accuracy:  {z0_acc:.1%}")
    print(f"Delta:                    {delta:+.1%}")
    print()
    
    if abs(delta) < 0.03:
        print("INTERPRETATION: Minimal accuracy drop. The token sequence (KV cache) is")
        print("sufficient to recover the reasoning. Subliminal information in the residual")
        print("stream is NOT load-bearing. The COT is faithful in the sense that what you")
        print("see (the text) is what the model uses to derive its answer.")
    elif abs(delta) < 0.10:
        print("INTERPRETATION: Moderate accuracy drop. Some subliminal information exists")
        print("in the residual stream that contributes to answer accuracy, but the text")
        print("content carries most of the signal.")
    else:
        print("INTERPRETATION: Significant accuracy drop. The residual stream carries")
        print("load-bearing hidden state that cannot be recovered from the text alone.")
        print("The COT is NOT fully faithful -- there is subliminal information.")
else:
    print("Run 03_interventions.ipynb with zeroed_layer_0 condition first.")